In [4]:
import os
import cv2
import numpy as np

# 1. Evita que o kernel morra por conflito de DLLs do OpenMP
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from ultralytics import YOLO

# 2. Carrega o modelo YOLO
modelo = YOLO("yolov8n.pt")

caminho_video = "videos/video_2.mp4"
if not os.path.exists(caminho_video):
    raise FileNotFoundError(f"Vídeo '{caminho_video}' não encontrado na pasta atual.")

cap = cv2.VideoCapture(caminho_video)

largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
if fps == 0:
    fps = 30

# 3. Configura o gravador de vídeo para salvar o resultado sem travar o Jupyter
caminho_saida = "resultado_contagem.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(caminho_saida, fourcc, fps, (largura, altura))

# 4. Linha de Contagem (75% da altura da tela, pegando a base das esteiras)
linha_y = int(altura * 0.75)

ids_contados = set()
total_caixas = 0
historico_posicoes = {}

print("Processando vídeo... O resultado será salvo em:", caminho_saida)
frame_idx = 0

while cap.isOpened():
    sucesso, frame = cap.read()
    if not sucesso:
        break

    frame_idx += 1

    # Inferência com rastreamento (persist=True)
    # classes=[24, 26, 28] ou sem filtro de classe para capturar pacotes e caixas
    resultados = modelo.track(frame, persist=True, verbose=False)

    if resultados[0].boxes is not None and resultados[0].boxes.id is not None:
        boxes = resultados[0].boxes.xyxy.cpu().numpy()
        track_ids = resultados[0].boxes.id.cpu().numpy().astype(int)
        confiancas = resultados[0].boxes.conf.cpu().numpy()

        for box, track_id, conf in zip(boxes, track_ids, confiancas):
            if conf < 0.25:
                continue

            x1, y1, x2, y2 = map(int, box)
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            # Desenha a caixa e centróide
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 120, 0), 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            # Lógica de contagem na descida
            if track_id in historico_posicoes:
                y_anterior = historico_posicoes[track_id]
                if y_anterior <= linha_y and cy >= linha_y:
                    if track_id not in ids_contados:
                        ids_contados.add(track_id)
                        total_caixas += 1

            historico_posicoes[track_id] = cy

    # Desenho da linha de corte
    cv2.line(frame, (0, linha_y), (largura, linha_y), (0, 0, 255), 2)
    cv2.putText(frame, "LINHA DE CONTAGEM", (20, linha_y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    # Painel de métricas
    cv2.rectangle(frame, (15, 20), (340, 85), (0, 0, 0), -1)
    cv2.putText(frame, f"TOTAL CAIXAS: {total_caixas}", (30, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

    # Grava o frame no arquivo de vídeo
    out.write(frame)

    if frame_idx % 60 == 0:
        print(f"Frame {frame_idx} processado | Caixas contadas até agora: {total_caixas}")

cap.release()
out.release()
print(f"\nFinalizado com sucesso! Total de caixas registradas: {total_caixas}")
print(f"Vídeo gravado em: {os.path.abspath(caminho_saida)}")

Processando vídeo... O resultado será salvo em: resultado_contagem.mp4
Frame 60 processado | Caixas contadas até agora: 0
Frame 120 processado | Caixas contadas até agora: 0
Frame 180 processado | Caixas contadas até agora: 0
Frame 240 processado | Caixas contadas até agora: 0
Frame 300 processado | Caixas contadas até agora: 0
Frame 360 processado | Caixas contadas até agora: 0
Frame 420 processado | Caixas contadas até agora: 0
Frame 480 processado | Caixas contadas até agora: 0
Frame 540 processado | Caixas contadas até agora: 0
Frame 600 processado | Caixas contadas até agora: 0
Frame 660 processado | Caixas contadas até agora: 0
Frame 720 processado | Caixas contadas até agora: 0
Frame 780 processado | Caixas contadas até agora: 0
Frame 840 processado | Caixas contadas até agora: 0
Frame 900 processado | Caixas contadas até agora: 0
Frame 960 processado | Caixas contadas até agora: 0
Frame 1020 processado | Caixas contadas até agora: 0
Frame 1080 processado | Caixas contadas até a